In [ ]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer, Float
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

load_dotenv()

#Si: Permissible limit, V0: Ideal value
WQI_STANDARDS = {
    'Θολότητα NTU': {'Si': 1.0, 'V0': 0, 'en_name': 'turbidity'},
    'Αργίλιο': {'Si': 200.0, 'V0': 0, 'en_name': 'aluminum'},
    'Αγωγιμότητα': {'Si': 2500.0, 'V0': 0, 'en_name': 'conductivity'},
    'Χλωριούχα': {'Si': 250.0, 'V0': 0, 'en_name': 'chlorides'},
    'Συγκέντρωση ιόντων υδρογόνου': {'Si': 8.5, 'V0': 7.0, 'en_name': 'ph'}
}

MONTH_MAP = {
    'Ιανουάριος': 1, 'Φεβρουάριος': 2, 'Μάρτιος': 3, 'Απρίλιος': 4,
    'Μάιος': 5, 'Ιούνιος': 6, 'Ιούλιος': 7, 'Αύγουστος': 8,
    'Σεπτέμβριος': 9, 'Οκτώβριος': 10, 'Νοέμβριος': 11, 'Δεκέμβριος': 12
}

In [ ]:
def clean_greek_value(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()

    if 'ΔΠ' in val:
        return np.nan
    
    val = val.replace('<', '')
    
    if '^' in val:
        val = val.split('^')[0]
    val = val.replace(',', '.')
    
    try:
        return float(val)
    except ValueError:
        return np.nan

def calculate_monthly_wqi(row_data):
    sum_1_Si = 0
    available_params = []
    
    for gr_name, limits in WQI_STANDARDS.items():
        if gr_name in row_data and pd.notna(row_data[gr_name]):
            sum_1_Si += 1 / limits['Si']
            available_params.append(gr_name)
            
    if not available_params:
        return np.nan
        
    k = 1 / sum_1_Si
    Wi_sum = 0
    WiQi_sum = 0
    
    for param in available_params:
        Vi = row_data[param]
        Si = WQI_STANDARDS[param]['Si']
        V0 = WQI_STANDARDS[param]['V0']
        
        Wi = k / Si
        Qi = ((Vi - V0) / (Si - V0)) * 100
        
        Wi_sum += Wi
        WiQi_sum += Wi * Qi
        
    if Wi_sum > 0:
        return round(WiQi_sum / Wi_sum, 2)
    return np.nan

def process_municipality(file_list, municipality_name):
    all_raw_data = []
    
    for file_path in file_list:
        sheets_dict = pd.read_excel(file_path, sheet_name=None)
        
        for sheet_name, df in sheets_dict.items():
            if df.empty:
                continue
                
            df['Cleaned_Value'] = df['Τιμή'].apply(clean_greek_value)
            
            if 'Month' in df.columns:
                df['Month'] = df['Month'].astype(str).str.strip()
                
            pivot_df = df.pivot_table(
                index=['Year', 'Month'], 
                columns='Φυσικοχημικές Παράμετροι', 
                values='Cleaned_Value', 
                aggfunc='mean'
            ).reset_index()
            
            all_raw_data.append(pivot_df)
            
    #Combines all regions and calculates the mean for the municipality
    combined_regions = pd.concat(all_raw_data, ignore_index=True)
    municipality_avg = combined_regions.groupby(['Year', 'Month']).mean().reset_index()
    
    municipality_avg['WQI'] = municipality_avg.apply(calculate_monthly_wqi, axis=1)
    
    final_df = pd.DataFrame({
        'municipality': municipality_name,
        'year': municipality_avg['Year'],
        'month': municipality_avg['Month'].map(MONTH_MAP),
        'wqi_score': municipality_avg['WQI'],
        'ph': municipality_avg.get('Συγκέντρωση ιόντων υδρογόνου', np.nan),
        'chlorides': municipality_avg.get('Χλωριούχα', np.nan),
        'turbidity': municipality_avg.get('Θολότητα NTU', np.nan),
        'aluminum': municipality_avg.get('Αργίλιο', np.nan),
        'conductivity': municipality_avg.get('Αγωγιμότητα', np.nan)
    })
    
    numeric_cols = ['ph', 'chlorides', 'turbidity', 'aluminum', 'conductivity']
    final_df[numeric_cols] = final_df[numeric_cols].round(3)
    
    #Sorts chronologically
    final_df = final_df.sort_values(by=['year', 'month']).reset_index(drop=True)
    
    return final_df

In [ ]:
municipality_configs = [
    {
        'name': 'Delta',
        'files': ['./data/water/delta/bipeth.xlsx']
    },
    {
        'name': 'Pylaia-Chortiatis',
        'files': [
            './data/water/panorama/panorama.xlsx',
            './data/water/pylaia/konstantinoypolitika.xlsx',
            './data/water/pylaia/pylaia.xlsx',
            './data/water/pylaia/pylaia_ikea.xlsx'
        ]
    },
    {
        'name': 'Ampelokipoi-Menemeni',
        'files': [
            './data/water/ampelokipoi/ampelokipoi.xlsx',
            './data/water/ampelokipoi/menemeni.xlsx',
            './data/water/ampelokipoi/dendropotamos.xlsx'
        ]
    },
    {
        'name': 'Kordelio-Evosmos',
        'files': [
            './data/water/kordelio/kordelio.xlsx',
            './data/water/kordelio/eyosmos.xlsx',
            './data/water/kordelio/dialogi.xlsx'
        ]
    },
    {
        'name': 'Kalamaria',
        'files': [
            './data/water/kalamaria/agios_ioannis_0.xlsx',
            './data/water/kalamaria/agios_panteleimonas_0.xlsx',
            './data/water/kalamaria/aretsoy_0.xlsx',
            './data/water/kalamaria/botsi_0.xlsx',
            './data/water/kalamaria/foinikas_0.xlsx',
            './data/water/kalamaria/kalamaria_0.xlsx',
            './data/water/kalamaria/karampoyrnaki_0.xlsx',
            './data/water/kalamaria/kifisia_0.xlsx',
            './data/water/kalamaria/nea_krini_0.xlsx'
        ]
    },
    {
        'name': 'Neapoli-Sykies',
        'files': [
            './data/water/neapoli/agios_paylos_0.xlsx',
            './data/water/neapoli/neapoli_0.xlsx',
            './data/water/neapoli/peyka.xlsx',
            './data/water/neapoli/sykies_0.xlsx'
        ]
    },
    {
        'name': 'Pavlos Melas',
        'files': [
            './data/water/pavlos melas/anthokipoi.xlsx',
            './data/water/pavlos melas/eykarpia.xlsx',
            './data/water/pavlos melas/ilioypoli.xlsx',
            './data/water/pavlos melas/meteora.xlsx',
            './data/water/pavlos melas/nikopoli.xlsx',
            './data/water/pavlos melas/polihni.xlsx',
            './data/water/pavlos melas/stayroypoli.xlsx'
        ]
    },
    {
        'name': 'Thessaloniki',
        'files': [
            './data/water/thessaloniki/40_ekklisies.xlsx',
            './data/water/thessaloniki/analipsi.xlsx',
            './data/water/thessaloniki/ano_poli.xlsx',
            './data/water/thessaloniki/ano_toympa.xlsx',
            './data/water/thessaloniki/deth-hanth.xlsx',
            './data/water/thessaloniki/harilaoy.xlsx',
            './data/water/thessaloniki/kato_toympa.xlsx',
            './data/water/thessaloniki/kentro_polis.xlsx',
            './data/water/thessaloniki/nea_paralia.xlsx',
            './data/water/thessaloniki/ntepo.xlsx',
            './data/water/thessaloniki/ntepo_0.xlsx',
            './data/water/thessaloniki/panagia_faneromeni.xlsx',
            './data/water/thessaloniki/plateia_dimokratias.xlsx',
            './data/water/thessaloniki/sfageia.xlsx',
            './data/water/thessaloniki/sholi_tyflon.xlsx',
            './data/water/thessaloniki/triandria.xlsx',
            './data/water/thessaloniki/xirokrini.xlsx'
        ]
    }
]

all_municipalities_results = []

for config in municipality_configs:
    result_df = process_municipality(config['files'], config['name'])
    all_municipalities_results.append(result_df)

final_db_dataset = pd.concat(all_municipalities_results, ignore_index=True)

In [ ]:
sync_db_url = os.getenv("DB_URL").replace("+asyncpg", "")
engine = create_engine(sync_db_url)
metadata = MetaData()

historical_wqi_table = Table(
    'historical_wqi', metadata,
    Column('municipality', String, primary_key=True),
    Column('year', Integer, primary_key=True),
    Column('month', Integer, primary_key=True),
    Column('wqi_score', Float),
    Column('ph', Float),
    Column('chlorides', Float),
    Column('turbidity', Float),
    Column('aluminum', Float),
    Column('conductivity', Float)
)
metadata.create_all(engine)

def insert_on_conflict_update(table, conn, keys, data_iter):
    data = [dict(zip(keys, row)) for row in data_iter]

    insert_stmt = insert(table.table).values(data)
    
    update_dict = {
        c.name: c for c in insert_stmt.excluded 
        if c.name not in ['municipality', 'year', 'month']
    }

    upsert_stmt = insert_stmt.on_conflict_do_update(
        index_elements=['municipality', 'year', 'month'],
        set_=update_dict
    )

    result = conn.execute(upsert_stmt)
    return result.rowcount

try:
    cleaned_df = final_db_dataset.replace({np.nan: None})
    
    rows_affected = cleaned_df.to_sql(
        'historical_wqi',
        engine,
        if_exists='append',
        index=False,
        method=insert_on_conflict_update
    )
except Exception as e:
    print(f"Error during database insertion: {e}")